# SCARA: kinematics and velocity

This notebook presents forward kinematics, inverse kinematics, the Jacobian, and singularities for our four-axis SCARA. The final cell opens **one native `mujoco.viewer` window** with joint, target, and joint-velocity modes.

The notebook contains no explicit GLFW imports or calls. Python calculates our kinematics; MuJoCo loads the model and displays the resulting poses. This is a kinematic visualization using `mj_forward`, with direct joint posing. Pickup/release uses geometric attachment to the tool, without contact-force or actuator dynamics. This notebook does not demonstrate inverse velocity kinematics.

![Our SCARA model](../figures/model_preview.png)

**Run locally:** open this file in VS Code with its Jupyter support, select the project's `.venv/Scripts/python.exe` kernel, and run the cells in order. This environment already has `ipykernel`. On a fresh setup, install the project requirements plus `notebooks/requirements.txt`. The viewer opens as a desktop window, so a hosted/headless notebook service will not provide this interactive display. The final cell stays busy until you close that window.

In [ ]:
from pathlib import Path
from collections import deque
from queue import SimpleQueue, Empty
import json
import math
import sys
import time

import numpy as np
import mujoco
import mujoco.viewer

# Works when the notebook is opened from the repository or notebooks folder.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "config" / "robot.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the SCARA project folder.")
sys.path.insert(0, str(ROOT / "scripts" / "kinematics"))
sys.path.insert(0, str(ROOT / "scripts" / "interactive"))
from target_workpiece import TargetWorkpiece
from scara_kinematics import (
    forward_kinematics, forward_transform, inverse_kinematics,
    task_jacobian, spatial_jacobian, is_singular, wrap_to_pi,
    UnreachableTargetError, JointLimitError,
)

config = json.loads((ROOT / "config" / "robot.json").read_text())
np.set_printoptions(precision=5, suppress=True)
print("MuJoCo:", mujoco.__version__)
print("Project:", ROOT)

## Model and conventions

The arm joint order is $q=[q_1,q_2,q_3,q_4]$: shoulder, elbow, downward slide, and downward-axis wrist rotation. Rotary coordinates use radians and $q_3$ uses metres. The two finger joints are separate from these four arm coordinates; G opens/closes them in the viewer.

We use standard DH: $A_i=R_z(\theta_i)T_z(d_i)T_x(a_i)R_x(\alpha_i)$.

| Link | $a_i$ | $\alpha_i$ | $d_i$ | $\theta_i$ |
| --- | --- | --- | --- | --- |
| 1 | $a_1$ | $0$ | $0$ | $q_1$ |
| 2 | $a_2$ | $\pi$ | $0$ | $q_2$ |
| 3 | $0$ | $0$ | $d_{3,\min}+q_3$ | $0$ |
| 4 | $0$ | $0$ | $d_4$ | $q_4$ |

World +Z points up. The shoulder frame is at world height $h$. The DH twist $\alpha_2=\pi$ points the final frames down. The MuJoCo construction frames and DH frames therefore have different axis orientations at the tool; positions agree. All dimensions come from `config/robot.json`.

In [2]:
for name in ["base_height_m", "a1_m", "a2_m", "d3_min_m", "d3_max_m", "d4_m"]:
    print(f"{name:18s} = {config[name]:.3f} m")

model = mujoco.MjModel.from_xml_path(str(ROOT / "models" / "scara.xml"))
data = mujoco.MjData(model)
q_example = np.array([math.radians(25), math.radians(-65), .045, math.radians(20)])
data.qpos[:4] = q_example
data.qpos[4:6] = config["finger_travel_m"]
mujoco.mj_forward(model, data)

base_height_m      = 0.280 m
a1_m               = 0.240 m
a2_m               = 0.180 m
d3_min_m           = 0.080 m
d3_max_m           = 0.190 m
d4_m               = 0.075 m


## 1. Forward kinematics

With $q_{12}=q_1+q_2$ and $d_3^*=d_{3,\min}+q_3$:

$$
x=a_1\cos q_1+a_2\cos q_{12},\qquad
y=a_1\sin q_1+a_2\sin q_{12},\qquad
z=h-d_3^*-d_4,\qquad
\phi=q_1+q_2-q_4.
$$

The functions below evaluate our equations from `scripts/kinematics/scara_kinematics.py`. MuJoCo supplies an independent position check.

In [3]:
pose = forward_kinematics(q_example)
print("Tool [x, y, z] (m):", pose[:3])
print("Tool yaw (deg):", math.degrees(pose[3]))
print("DH transform in world coordinates:\n", forward_transform(q_example))
error = np.linalg.norm(pose[:3] - data.site("tool").xpos)
assert error < 1e-10
print("Position difference from MuJoCo (m):", error)

Tool [x, y, z] (m): [ 0.3554  -0.01427  0.08   ]
Tool yaw (deg): -60.00000000000001
DH transform in world coordinates:
 [[ 0.5     -0.86603  0.       0.3554 ]
 [-0.86603 -0.5      0.      -0.01427]
 [ 0.       0.      -1.       0.08   ]
 [ 0.       0.       0.       1.     ]]
Position difference from MuJoCo (m): 8.092071311423672e-17


## 2. Inverse kinematics

For a target $(x,y,z,\phi)$, both elbow branches follow from

$$
q_2=\pm\cos^{-1}\left(\frac{x^2+y^2-a_1^2-a_2^2}{2a_1a_2}\right),\qquad
q_1=\operatorname{atan2}(y,x)-\operatorname{atan2}(a_2\sin q_2,a_1+a_2\cos q_2).
$$

$$
q_3=h-z-d_4-d_{3,\min},\qquad q_4=q_1+q_2-\phi.
$$

Horizontal geometric reach requires $|a_1-a_2|\le\sqrt{x^2+y^2}\le a_1+a_2$. The code also checks the slide range and rotary joint limits. These are kinematic checks; they do not certify collision-free motion.

In [4]:
target_example = np.array([.30, .02, .075, .1])
solutions = inverse_kinematics(target_example)
for number, q in enumerate(solutions, 1):
    print(f"Branch {number}: q1={math.degrees(q[0]):.2f} deg, "
          f"q2={math.degrees(q[1]):.2f} deg, q3={q[2]*1000:.2f} mm, "
          f"q4={math.degrees(q[3]):.2f} deg")
    assert np.allclose(forward_kinematics(q), target_example, atol=1e-10)

try:
    inverse_kinematics([.60, 0, .075, 0])
except UnreachableTargetError as error:
    print("Unreachable example:", error)

Branch 1: q1=-32.96 deg, q2=89.73 deg, q3=50.00 mm, q4=51.04 deg
Branch 2: q1=40.59 deg, q2=-89.73 deg, q3=50.00 mm, q4=-54.88 deg
Unreachable example: Horizontal radius 0.600000 m is outside [0.060000, 0.420000] m.


## 3. Forward velocity and singularities

The task Jacobian maps joint rates to tool velocity, $\dot{x}=J(q)\dot{q}$, for $x=[x,y,z,\phi]^T$:

$$
J=\begin{bmatrix}
-a_1\sin q_1-a_2\sin q_{12} & -a_2\sin q_{12} & 0 & 0\\
a_1\cos q_1+a_2\cos q_{12} & a_2\cos q_{12} & 0 & 0\\
0&0&-1&0\\
1&1&0&-1
\end{bmatrix}.
$$

Its determinant is $a_1a_2\sin q_2$. Rank is lost at $\sin q_2=0$. The fully folded case lies outside our configured elbow range, while the straight arm is permitted. This notebook selects joint rates and computes tool velocity; it does not solve the inverse velocity problem.

In [5]:
qdot_example = np.array([.2, -.15, .012, .1])  # rad/s, rad/s, m/s, rad/s
J = task_jacobian(q_example)
tool_velocity = J @ qdot_example
print("Task Jacobian:\n", J)
print("Tool [vx, vy, vz, yaw_rate] (SI):", tool_velocity)

dt = 1e-6
numeric = (forward_kinematics(q_example + dt*qdot_example)
           - forward_kinematics(q_example - dt*qdot_example)) / (2*dt)
assert np.allclose(tool_velocity, numeric, atol=1e-8)
assert is_singular([0, 0, .045, 0])
print("Velocity finite-difference check: PASS")
print("Straight-arm Jacobian rank:", np.linalg.matrix_rank(task_jacobian([0, 0, .045, 0])))

# Compare the 6x4 geometric Jacobian with MuJoCo at the same pose.
jp, jr = np.zeros((3, model.nv)), np.zeros((3, model.nv))
mujoco.mj_jacSite(model, data, jp, jr, model.site("tool").id)
assert np.allclose(spatial_jacobian(q_example), np.vstack((jp[:, :4], jr[:, :4])), atol=1e-10)
print("MuJoCo geometric-Jacobian comparison: PASS")

Task Jacobian:
 [[ 0.01427  0.1157   0.       0.     ]
 [ 0.3554   0.13789  0.       0.     ]
 [ 0.       0.      -1.       0.     ]
 [ 1.       1.       0.      -1.     ]]
Tool [vx, vy, vz, yaw_rate] (SI): [-0.0145  0.0504 -0.012  -0.05  ]
Velocity finite-difference check: PASS
Straight-arm Jacobian rank: 3
MuJoCo geometric-Jacobian comparison: PASS


## 4. One interactive viewer

Run the next two definition cells, then the final launch cell. The window uses MuJoCo's own viewer and text overlays. No custom desktop window, buttons, or rendering loop is created here.

| Key | Action |
| --- | --- |
| **F8** | Cycle Joints → Target → Velocity |
| **F9** | Select the next joint or target coordinate |
| **Left / Right** | Decrease / increase the selected value |
| **F10** | Joints: straight arm; Target: switch elbow branch; Velocity: zero rates |
| **F11** | Target: start/stop an IK circle; Velocity: run/pause |
| **F12** | Reset the scene |
| **P** | Align open jaws with the connector using IK |
| **G** | Close/pick up or open/release |

The connector stays visible in every mode. Press P then G to pick it up, and jog the joints or run joint velocities to carry it. G releases it at the current pose. Closing away from the connector misses. This uses kinematic attachment, without gravity or grasp forces. The Target controls move the tool target; a free connector stays put. Close the viewer window to finish the cell.


In [ ]:
class NotebookDemo:
    """Plain kinematic state driven by MuJoCo viewer key events."""
    def __init__(self):
        self.model = mujoco.MjModel.from_xml_path(str(ROOT / "models" / "scara_kinematics.xml"))
        self.data = mujoco.MjData(self.model)
        self.workpiece = TargetWorkpiece(self.model, self.data)
        self.limits = self.model.jnt_range[:4].copy()
        self.grip_open = True
        self.mode = 0
        self.selected = 0
        self.commands = SimpleQueue()
        self.trail = deque(maxlen=180)
        self.rates = np.zeros(4)
        self.actual_rates = np.zeros(4)
        rate_config = config["velocity_limit"]
        self.max_rates = np.minimum(
            [rate_config["shoulder_rad_s"], rate_config["elbow_rad_s"],
             rate_config["vertical_m_s"], rate_config["wrist_rad_s"]],
            [math.radians(60), math.radians(60), .06, math.radians(90)])
        self.home()

    @property
    def q(self):
        return self.data.qpos[:4].copy()

    @property
    def pose(self):
        return forward_kinematics(self.q)

    def set_q(self, q):
        self.data.qpos[:4] = np.clip(q, self.limits[:, 0], self.limits[:, 1])
        self.data.qpos[4:6] = (config["finger_travel_m"] if self.grip_open else
                                 self.workpiece.finger_positions if self.workpiece.held else 0)
        self.data.qvel[:] = 0
        mujoco.mj_forward(self.model, self.data)
        self.workpiece.follow(self.pose)
        point = self.pose[:3]
        if not self.trail or np.linalg.norm(point - self.trail[-1]) > .001:
            self.trail.append(point.copy())

    def home(self):
        self.grip_open = True
        self.workpiece.reset()
        self.running = False
        self.valid = True
        self.status = "Ready"
        self.branch = -1
        self.phase = 0.
        self.rates[:] = 0
        self.actual_rates[:] = 0
        self.trail.clear()
        self.set_q(q_example)
        self.target = self.pose.copy()

    def toggle_grip(self):
        self.running = False
        self.actual_rates[:] = 0
        self.grip_open = not self.grip_open
        if self.grip_open:
            held = self.workpiece.held
            self.workpiece.release()
            self.status = "Released at current pose" if held else "Gripper open"
        elif self.workpiece.try_grasp(self.pose):
            self.status = "Connector held: move joints to carry it"
        else:
            self.status = "Missed: align jaws with connector"
        self.set_q(self.q)

    def go_to_object(self):
        self.running = False
        self.actual_rates[:] = 0
        if self.workpiece.held:
            self.status = "Already holding connector"
            return
        self.grip_open = True
        self.set_q(self.q)
        self.target = self.workpiece.pose.copy()
        self.solve_target()
        if self.valid:
            self.status = "Aligned: close gripper to pick up"

    def solve_target(self):
        try:
            options = inverse_kinematics(self.target)
            choices = [q for q in options if q[1] * self.branch >= -1e-9]
            if not choices:
                self.valid = False
                self.status = "Chosen elbow branch violates a joint limit"
                return
            self.set_q(choices[0])
            self.valid = True
            self.status = "Target reached"
        except (UnreachableTargetError, JointLimitError) as error:
            self.valid = False
            self.status = "Unreachable target" if isinstance(error, UnreachableTargetError) else "Joint limit excludes target"

    def on_key(self, keycode):
        # MuJoCo viewer supplies key codes. The callback only queues an action;
        # model changes happen in the main loop under viewer.lock().
        names = {297: "mode", 298: "select", 263: "minus", 262: "plus",
                 299: "special", 300: "run", 301: "home", 71: "grip", 80: "pickup"}
        if keycode in names:
            self.commands.put(names[keycode])

    def handle(self, action):
        self.actual_rates[:] = 0
        if action == "home":
            self.home()
        elif action == "grip":
            self.toggle_grip()
        elif action == "pickup":
            self.go_to_object()
        elif action == "mode":
            self.mode = (self.mode + 1) % 3
            self.running = False
            self.rates[:] = 0
            self.target = self.pose.copy()
            self.branch = 1 if self.q[1] >= 0 else -1
            self.valid = True
            self.status = "Ready"
            self.trail.clear()
        elif action == "select":
            self.selected = (self.selected + 1) % 4
        elif action in ("minus", "plus"):
            sign = -1 if action == "minus" else 1
            i = self.selected
            if self.mode == 0:
                q = self.q
                q[i] += sign * [math.radians(2), math.radians(2), .002, math.radians(2)][i]
                self.set_q(q)
            elif self.mode == 1:
                self.running = False
                self.target[i] += sign * [.005, .005, .005, math.radians(5)][i]
                self.solve_target()
            else:
                self.rates[i] = np.clip(self.rates[i] + sign * [math.radians(5), math.radians(5), .005, math.radians(5)][i],
                                        -self.max_rates[i], self.max_rates[i])
        elif action == "special":
            if self.mode == 0:
                self.set_q([0, 0, .045, 0])
                self.status = "Straight arm: singular"
            elif self.mode == 1:
                self.running = False
                self.branch *= -1
                self.trail.clear()
                self.solve_target()
            else:
                self.rates[:] = 0
                self.running = False
                self.status = "Rates zeroed"
        elif action == "run" and self.mode in (1, 2):
            self.running = not self.running
            self.status = "Running" if self.running else "Paused"

    def tick(self, dt):
        self.actual_rates[:] = 0
        if self.running and self.mode == 2 and dt > 0:
            before = self.q
            desired = before + dt * self.rates
            self.set_q(desired)
            self.actual_rates = (self.q - before) / dt
            self.status = "Joint limit: motion clipped" if not np.allclose(self.q, desired, atol=1e-12, rtol=0) else "Running"
        elif self.running and self.mode == 1:
            self.phase += .55 * dt
            self.target[:] = [.29 + .035*math.cos(self.phase), .035*math.sin(self.phase), .08, 0]
            self.solve_target()

    def overlay(self):
        names = ["JOINTS / forward kinematics", "TARGET / inverse kinematics", "VELOCITY / J qdot"]
        if self.mode == 1:
            values = self.target * [1000, 1000, 1000, 180/math.pi]
            labels = ["Target X (mm)", "Target Y (mm)", "Target Z (mm)", "Target yaw (deg)"]
        else:
            values = (self.rates if self.mode == 2 else self.q) * [180/math.pi, 180/math.pi, 1000, 180/math.pi]
            units = ["deg/s", "deg/s", "mm/s", "deg/s"] if self.mode == 2 else ["deg", "deg", "mm", "deg"]
            labels = [f"{n} ({u})" for n, u in zip(["Shoulder", "Elbow", "Slide", "Wrist"], units)]
        lines = ["SCARA | " + names[self.mode], ""]
        lines += [f"{'>' if i == self.selected else ' '} {label}: {value: .1f}" for i, (label, value) in enumerate(zip(labels, values))]
        p = self.pose
        lines += ["", f"Tool mm: {p[0]*1000:.1f}, {p[1]*1000:.1f}, {p[2]*1000:.1f}",
                  f"Yaw: {math.degrees(p[3]):.1f} deg"]
        if self.mode == 1:
            q = self.q * [180/math.pi, 180/math.pi, 1000, 180/math.pi]
            lines += [f"q1/q2: {q[0]:.1f}, {q[1]:.1f} deg", f"q3/q4: {q[2]:.1f} mm, {q[3]:.1f} deg"]
        elif self.mode == 2:
            v = task_jacobian(self.q) @ self.actual_rates
            lines += [f"V mm/s: {v[0]*1000:.1f}, {v[1]*1000:.1f}, {v[2]*1000:.1f}",
                      f"Yaw rate: {math.degrees(v[3]):.1f} deg/s"]
        measure = abs(math.sin(self.q[1]))
        label = "SINGULAR" if measure < 1e-8 else "Near singular" if measure < .12 else "Regular"
        grip = "held" if self.workpiece.held else "free"
        lines += ["", f"{label} | abs(sin q2) = {measure:.3f}",
                  f"Connector: {grip} | Gripper: {'open' if self.grip_open else 'closed'}", self.status]
        return "\n".join(lines)

In [ ]:
def add_marker(scene, kind, start, size, color, end=None):
    if scene.ngeom >= scene.maxgeom:
        return
    geom = scene.geoms[scene.ngeom]
    mujoco.mjv_initGeom(geom, kind, np.asarray(size, dtype=float),
                       np.asarray(start, dtype=float), np.eye(3).ravel(),
                       np.asarray(color, dtype=np.float32))
    if end is not None:
        mujoco.mjv_connector(geom, kind, size[0], np.asarray(start, dtype=float), np.asarray(end, dtype=float))
    scene.ngeom += 1


def update_markers(demo, scene):
    scene.ngeom = 0
    points = list(demo.trail)
    for a, b in zip(points, points[1:]):
        add_marker(scene, mujoco.mjtGeom.mjGEOM_LINE, a, [2, 0, 0], [.2, .7, .75, 1], b)
    p = demo.pose[:3]
    for axis, color in zip(np.eye(3), [(1,.2,.2,1), (.2,.8,.3,1), (.2,.4,1,1)]):
        add_marker(scene, mujoco.mjtGeom.mjGEOM_ARROW, p, [.002,0,0], color, p+.04*axis)
    if demo.mode == 1:
        color = [.15,.85,.3,.85] if demo.valid else [1,.15,.1,.9]
        start, end = demo.workpiece.yaw_arrow(demo.target)
        add_marker(scene, mujoco.mjtGeom.mjGEOM_ARROW, start, [.002,0,0], color, end)
    if demo.mode == 2:
        velocity = (task_jacobian(demo.q) @ demo.actual_rates)[:3]
        if np.linalg.norm(velocity) > 1e-6:
            add_marker(scene, mujoco.mjtGeom.mjGEOM_ARROW, p, [.003,0,0], [1,.75,.15,1], p+.7*velocity)


def run_viewer(duration=None):
    """Open MuJoCo's native viewer; duration is optional for timed checks."""
    demo = NotebookDemo()
    instructions = ("F8 mode | F9 select | Left - | Right +\n"
                    "F10 straight / elbow / zero | F11 run-pause | F12 reset\n"
                    "P go to object | G close-pickup / open-release\n"
                    "Direct joint posing | no dynamics stepping | Close window to return")
    with mujoco.viewer.launch_passive(demo.model, demo.data, key_callback=demo.on_key,
                                     show_left_ui=False, show_right_ui=False) as viewer:
        with viewer.lock():
            viewer.cam.lookat[:] = [.17, 0, .17]
            viewer.cam.distance = .85
            viewer.cam.azimuth = 115
            viewer.cam.elevation = -25
            viewer.opt.sitegroup[:] = 0
        started = previous = time.perf_counter()
        try:
            while viewer.is_running():
                now = time.perf_counter()
                with viewer.lock():
                    while True:
                        try:
                            demo.handle(demo.commands.get_nowait())
                        except Empty:
                            break
                    demo.tick(min(now-previous, .04))
                    viewer.opt.geomgroup[3] = 1
                    update_markers(demo, viewer.user_scn)
                    readout = demo.overlay()
                previous = now
                viewer.set_texts([
                    (mujoco.mjtFontScale.mjFONTSCALE_100, mujoco.mjtGridPos.mjGRID_TOPLEFT, readout, ""),
                    (mujoco.mjtFontScale.mjFONTSCALE_100, mujoco.mjtGridPos.mjGRID_BOTTOMLEFT, instructions, ""),
                ])
                viewer.sync()
                if duration is not None and now-started >= duration:
                    break
                time.sleep(max(0, 1/60-(time.perf_counter()-now)))
        finally:
            demo.running = False
            demo.actual_rates[:] = 0
    return demo

Run this cell to open the viewer. Press **P** to align with the object, **G** to pick up, then use **F9** and **Left/Right** to jog joints. **G** releases; **F12** resets. **F8** switches modes. Close the window before restarting this cell or the kernel.


In [ ]:
demo = run_viewer()
print("Viewer closed. The notebook can continue.")